# Feature engineering → XGBoost tuning → calibration
Real saved results. This notebook reads evidence; it does not train again or retune on the reused holdout.

In [1]:
from pathlib import Path
import json
EXPERIMENT = Path.cwd() if (Path.cwd()/'protocol.json').exists() else globals().get('EXPERIMENT')
assert EXPERIMENT is not None, 'Set EXPERIMENT to the experiment directory.'
load = lambda name: json.loads((EXPERIMENT/name).read_text())
print('Seed:', load('protocol.json')['seed'])
print('Trials:', len(list((EXPERIMENT/'trials').glob('*.json'))))

Seed: 202610231
Trials: 50


## Preprocessing
Original: standardized numeric + one-hot categorical. Ordered: numeric ordered scales, other categories one-hot (drop binary reference). Engineered: seven derived fields. Every learned value is fitted within a training fold. See `tools/feature_pipeline.py` for implementation.

In [2]:
d=load('feature_selection.json')
for mode,row in d['comparisons'].items(): print(mode, row['dimensions'], 'columns; CV logloss:', round(row['mean_log_loss'],6))
print('Selected:', d['selected'])

original 69 columns; CV logloss: 0.71974
ordered 31 columns; CV logloss: 0.719603
engineered 38 columns; CV logloss: 0.719496
Selected: engineered


In [3]:
import sys
import pandas as pd
sys.path.insert(0,str(EXPERIMENT.parents[1]/'tools'))
from feature_pipeline import FeatureEncoder
encoder=FeatureEncoder.restore(load('encoder.json'))
frame=pd.read_csv(EXPERIMENT/'backup/data/expanded_survey_dataset/respondents.csv')
examples=frame.loc[frame.split=='train'].head(3)
x=encoder.transform(examples)
print('Three original respondents -> shape:',x.shape)
print('Feature order:',encoder.names)
print('Training-fitted BMI clipping bounds:',encoder.bmi_limits)
if encoder.mode=='engineered': print(pd.DataFrame(x[:,-7:],columns=encoder.names[-7:]).to_string(index=False))
del frame

Three original respondents -> shape: (3, 38)
Feature order: ['age', 'sleep_duration', 'unhealthy_days', 'bmi', 'sex_2', 'exercise_2', 'general_health', 'marital_1', 'marital_2', 'marital_3', 'marital_4', 'marital_5', 'marital_6', 'education', 'employment_1', 'employment_2', 'employment_3', 'employment_4', 'employment_5', 'employment_6', 'employment_7', 'employment_8', 'income', 'support', 'isolation', 'job_loss_2', 'food_insecurity', 'bill_difficulty_2', 'utility_threat_2', 'transport_barrier_2', 'medical_cost_2', 'social_risk_count', 'unhealthy_14plus', 'unhealthy_log1p', 'sleep_under7', 'sleep_7to9', 'sleep_over9', 'bmi_clipped']
Training-fitted BMI clipping bounds: [18.02, 49.60270000000004]
 social_risk_count  unhealthy_14plus  unhealthy_log1p  sleep_under7  sleep_7to9  sleep_over9  bmi_clipped
               0.0               0.0         0.000000           1.0         0.0          0.0    26.570000
               1.0               0.0         1.098612           1.0         0.0     

## Split protection
Search/fitting and calibration roles are disjoint. Grouped folds prevent identical feature profiles straddling partitions. Validation selects the final cutoff; the historically reused test is evaluated once after freezing.

In [4]:
import csv
from collections import Counter
roles=list(csv.DictReader((EXPERIMENT/'training_roles.csv').open()))
print(Counter(row['role'] for row in roles))
print('Unique respondents:', len({row['respondent_id'] for row in roles}))

Counter({'search_and_fit': 115416, 'calibration_only': 12832})
Unique respondents: 128248


## Broader tuning
Two grouped outer folds; inner early stopping. Unweighted log-loss objective, not accuracy. Training-only class-weight exponent is tuned. No fabricated data or validation resampling.

In [5]:
print(json.dumps(load('best_trial.json')['params'],indent=2))
print('Best CV logloss:', load('best_trial.json')['mean_log_loss'])

{
  "n_estimators": 700,
  "learning_rate": 0.06800981021604165,
  "max_depth": 2,
  "min_child_weight": 7.397930995578851,
  "subsample": 0.7464666411823344,
  "colsample_bytree": 0.8569248655919637,
  "reg_alpha": 0.005029298116217086,
  "reg_lambda": 3.4621066431849417,
  "gamma": 0.2747336593377415,
  "weight_power": 0.006129106017468772
}
Best CV logloss: 0.6842851938183006


## Calibration and decision rule
Temperature is fitted on one training-reserve half and checked on the other. It preserves argmax. Predict high if P(high) >= cutoff; otherwise choose the more probable lower class.

In [6]:
c=load('calibration.json')
for key in ['enabled','temperature','check_raw_logloss','check_calibrated_logloss','check_raw_brier','check_calibrated_brier']: print(key,c[key])
f=load('frozen_selection.json')
for name in ['xgb','ensemble']: print(name,'floor',f[name]['floor'],'cutoff',f[name]['candidate']['threshold'],'eligible',f[name]['candidate']['eligible'])

enabled False
temperature 1.0
check_raw_logloss 0.6885141577495567
check_calibrated_logloss 0.6885473507913963
check_raw_brier 0.39987460142367226
check_calibrated_brier 0.3998923245263452
xgb floor 0.6503653738055087 cutoff 0.19251422297188306 eligible False
ensemble floor 0.6003372681281619 cutoff 0.2750725590638555 eligible False


## Frozen reused-holdout results
Always-rarely baseline is included. A validation recall floor is not a guarantee for unseen respondents. Do not rerun selection after inspecting this table.

In [7]:
r=load('test_report.json')
for name in ['xgb','ensemble']:
    for version in ['original_actual','original_matched','candidate']:
        m=r[name][version]
        print(name,version,'accuracy',round(m['accuracy'],4),'high recall',round(m['recall'][2],4),'high missed',m['high_missed'])
print('Always rarely:',r['always_rarely'])
print('Gates passed:',r['approved_for_runtime_verification'])

xgb original_actual accuracy 0.6754 high recall 0.6533 high missed 1255
xgb original_matched accuracy 0.6767 high recall 0.6514 high missed 1262
xgb candidate accuracy 0.6875 high recall 0.6472 high missed 1277
ensemble original_actual accuracy 0.6839 high recall 0.6014 high missed 1443
ensemble original_matched accuracy 0.6836 high recall 0.6019 high missed 1441
ensemble candidate accuracy 0.6935 high recall 0.605 high missed 1430
Always rarely: {'accuracy': 0.6578227749202422, 'high_recall': 0.0}
Gates passed: False


In [8]:
print(json.dumps(load('gate_explanations.json'),indent=2))

{
  "xgb": [
    "No validation threshold passed all declared accuracy, recall, macro-F1, QWK and middle-recall gates; candidate threshold is a reporting-only fallback.",
    "Final high-stress recall below the current-model/60% floor.",
    "macro_f1 below current decision.",
    "qwk below current decision.",
    "Sometimes recall below current decision.",
    "macro_f1 below matched decision.",
    "qwk below matched decision.",
    "Sometimes recall below matched decision."
  ],
  "ensemble": [
    "No validation threshold passed all declared accuracy, recall, macro-F1, QWK and middle-recall gates; candidate threshold is a reporting-only fallback.",
    "macro_f1 below current decision.",
    "qwk below current decision.",
    "Sometimes recall below current decision.",
    "macro_f1 below matched decision.",
    "qwk below matched decision.",
    "Sometimes recall below matched decision."
  ]
}
